# Fitting a surface

`fit()` hands the `ModelSpec` to a backend. The default is the sampler-free Laplace
approximation — honest by construction: if the mode search does not converge or the Hessian at
the mode is not positive definite, the result is a typed `Unverified`, not a posterior. With the
`[numpyro]` extra, `backend="numpyro"` runs NUTS on the same tree through the jax interpreter.

In [ ]:
import numpy as np

from axiom.core import Interval, is_failure
from axiom.surface import (
    FitResult, GeometricCarryover, HillKernel, MarginalHorizon, Surface, counterfactual_doses, fit, marginal,
    marginal_expr, marginal_total, predict, predict_marginal,
)
from axiom.sim import surface_world

In [ ]:
world = surface_world(n_units=4, n_periods=30, treatments=("a",), kernels=HillKernel(reference_dose=50.0),
                      carryover=GeometricCarryover(max_lag=4), intercept="shared", noise_sd=0.5, seed=7)
print({k: np.round(v, 3) for k, v in world.theta.items()})
print(world.panel)

In [ ]:
result: FitResult = fit(world.spec, world.panel, backend="laplace", draws=2000, seed=0)
print("converged:", result.converged, "| provenance keys:", sorted(result.provenance)[:6])
post = result.posterior
if is_failure(post):
    print(post)
else:
    for name in ("beta_a", "k_a", "s_a", "lam_a", "sigma"):
        s = post.summary(name, definition="hdi", mass=0.9)
        iv: Interval = s.interval
        print(f"{name:7s} truth={float(world.theta[name]):8.3f} mean={s.mean:8.3f} {iv}")

## Prediction and marginal effects

`predict` evaluates the tree once per draw; `marginal` is the closed-form derivative of the
kernel through the carryover (same-period effect, `w_0 · f'`), `marginal_total` accumulates over
the carryover horizon. Both are expression trees — see `marginal_expr`.

In [ ]:
surface = Surface(world.spec)
if not is_failure(post):
    pred = predict(surface, post, world.data, seed=0)
    print(pred.values.shape, pred.intervention.version)
    print("coverage of the noise-free mean by the 90% HDI:",
          np.mean([(np.quantile(pred.values[..., u, t], 0.05) <= world.mean[u, t] <= np.quantile(pred.values[..., u, t], 0.95)) for u in range(4) for t in range(30)]).round(2))
from axiom.core import dimension, latex
print(dimension(marginal_expr(surface, "a")))
print(marginal(surface, world.theta, world.data, "a")[0, :4].round(4), marginal_total(surface, world.theta, world.data, "a")[0, :4].round(4))

`counterfactual_doses` applies an `Intervention` (set / scale / shift, optionally on a support
window) to the fitted dose arrays; `predict_marginal` gives per-draw marginal effects for a
chosen `MarginalHorizon` — `period` (same-period), `total` (over the carryover horizon), or
`shift` (the derivative of the windowed outcome w.r.t. a common shift of the support's doses).

In [ ]:
from axiom.core import Intervention, TimeWindow

cf = counterfactual_doses(surface, world.data, Intervention(doses={"a": 2.0}, mode="scale", window=TimeWindow(start=5, stop=10)))
print(cf["a"][0, 3:12].round(1), "<- scaled on [5,10) only")
if not is_failure(post):
    horizon: MarginalHorizon = "shift"
    pm = predict_marginal(surface, post, world.data, "a", horizon=horizon, support=TimeWindow(start=5, stop=10))
    print(pm.values.shape, pm.values.mean(axis=(0, 1))[0, 3:12].round(4))

## Unit invariance of a fit

Fit the same world expressed in cents (doses ×100): the shape `s` and carryover `lam` agree
within Monte-Carlo error, and the scale `k` differs by exactly the conversion factor.

In [ ]:
import pandas as pd

from axiom.core import D, Treatment
from axiom.data import Panel, RoleMap

frame = world.panel.frame
frame_cents = frame.assign(a=frame["a"] * 100)
roles = world.panel.roles
roles_cents = roles.model_copy(update={"treatments": {"a": Treatment(name="a", dimension=D.currency, unit="cents")}})
spec_cents = world.spec.model_copy(update={
    "treatments": (Treatment(name="a", dimension=D.currency, unit="cents"),),
    "kernels": {"a": HillKernel(reference_dose=5000.0)},
})
res_cents = fit(spec_cents, Panel(frame_cents, roles_cents), backend="laplace", draws=2000, seed=0)
pc = res_cents.posterior
if not (is_failure(post) or is_failure(pc)):
    for name in ("s_a", "lam_a", "k_a"):
        print(name, round(post.summary(name).mean, 3), round(pc.summary(name).mean, 3))